# Extracting the dataset from the zipfile, you don't have to run this again

In [1]:
import zipfile

In [2]:
def unzip_file(zip_filepath, extract_to_path):
    """
    Extracts all files from a zip archive.

    Args:
        zip_filepath (str): The path to the zip file.
        extract_to_path (str): The path to extract the contents to.
    """
    with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
        zip_ref.extractall(extract_to_path)

# Example usage:
zip_filepath = 'CelebA_HQ_face_gender_dataset.zip'
extract_to_path = 'gender_dataset'
unzip_file(zip_filepath, extract_to_path)

# Checking the baseline performance of our gender classifier on the entire dataset

In [ ]:
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import torch
# Load the model and processor
custom_cache_dir="../classifier_gender/classifier_model"
model_name = "rizvandwiki/gender-classification-2"
model = ViTForImageClassification.from_pretrained(model_name, cache_dir=custom_cache_dir)
processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=custom_cache_dir)

# 'gender_dataset/CelebA_HQ_face_gender_dataset/train/male'
image_path ="PATH OF SAMPLE IMAGE"
#[Failed for guy3]
image = Image.open(image_path).convert("RGB")  # Ensures it's 3-channel

# Preprocess the image and run inference
inputs = processor(image, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
    pred = logits.argmax(dim=1).item()

# Print prediction
label_map = model.config.id2label  # Usually: {0: 'female', 1: 'male'}
print("Predicted gender:", label_map[pred])

Predicted gender: female


# Male

In [ ]:
import os
from PIL import Image
from PIL import ImageOps 
import torch
from tqdm import tqdm

# Path to the image directory
image_dir = "PATH TO MALE IMAGE DIR"

# Get list of image files
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
image_files = image_files[:len(image_files) // 2]


# Counters
total = 0
correct = 0

# Loop with progress bar
from PIL import ImageOps
import matplotlib.pyplot as plt  # Add this import to display images

# Center crop size (adjust if you want)

for filename in tqdm(image_files, desc="Processing images"):
    image_path = os.path.join(image_dir, filename)
    try:
        image = Image.open(image_path).convert("RGB")
        
        # 👉 Manual Center Crop (no resize)
        width, height = image.size
        crop_width, crop_height = 650, 650  # or whatever size you want
        
        left = (width - crop_width) // 2
        top = (height - crop_height) // 2
        right = left + crop_width
        bottom = top + crop_height

        image = image.crop((left, top, right, bottom))  # PIL.Image.crop takes (left, top, right, bottom)

        inputs = processor(image, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = logits.argmax(dim=1).item()

        label_map = model.config.id2label
        predicted_label = label_map[pred]

        total += 1
        if predicted_label.lower() == "male":
            correct += 1
    except Exception as e:
        print(f"[Failed for {filename}]: {e}")

# Final accuracy
if total > 0:
    accuracy = correct / total
    print(f"\nAccuracy: {accuracy:.4f} ({correct}/{total})")
else:
    print("No images found or processed.")


Processing images: 100%|██████████| 4422/4422 [18:54<00:00,  3.90it/s]


Accuracy: 0.8476 (3748/4422)


# Female

In [ ]:
image_dir = "PATH TO FEMALE IMAGE DIR"

# Get list of image files
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

# Counters
total = 0
correct = 0

# Loop with progress bar
for filename in tqdm(image_files, desc="Processing images"):
    image_path = os.path.join(image_dir, filename)
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(image, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = logits.argmax(dim=1).item()

        label_map = model.config.id2label
        predicted_label = label_map[pred]

        total += 1
        if predicted_label.lower() == "female":
            correct += 1
    except Exception as e:
        print(f"[Failed for {filename}]: {e}")

# Final accuracy
if total > 0:
    accuracy = correct / total
    print(f"\nAccuracy: {accuracy:.4f} ({correct}/{total})")
else:
    print("No images found or processed.")

Processing images:  52%|█████▏    | 7862/15154 [37:32<34:48,  3.49it/s]  


KeyboardInterrupt: 

In [10]:
accuracy = (total -correct) / total
print(f"\nAccuracy: {accuracy:.4f} ({total -correct}/{total})")


Accuracy: 0.9994 (7857/7862)
